In [1]:
# Fix Windows symlink issue
import os
os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
print("✅ Disabled HuggingFace symlinks for Windows compatibility")

✅ Disabled HuggingFace symlinks for Windows compatibility


# RAG on ASQA Long-Form QA Dataset

This notebook implements and compares two RAG baselines on the ASQA dataset:
1. **Normal RAG**: Standard retrieve-and-generate pipeline
2. **RAG + LLM Filter**: Two-stage filtering (context + answer)

ASQA focuses on ambiguous questions requiring long-form answers (50-200 words).

## 1. Setup and Imports

In [2]:
import sys
sys.path.append('..')

import os
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import dotenv

# Load environment variables
dotenv.load_dotenv()

# Import RAG system components
from src.config import RAGConfig
from src.rag_system import RAGSystem
from src.data.loader import ASQALoader
from src.filtering.llm_filter import LLMFilterPipeline

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\filtering\llm_filter.py:17: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 2. Load ASQA Dataset

In [3]:
# Initialize ASQA loader
loader = ASQALoader()

# Load data
train_path = "../data/asqa/train.csv"
dev_path = "../data/asqa/dev.csv"

df_train, df_dev = loader.load_data(train_path, dev_path)

# Display statistics
stats = loader.get_statistics()
print("\nDataset Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Loading ASQA data...
Loaded 4353 training samples
Loaded 948 dev samples

Answer length statistics (words):
  Mean: 73.3
  Median: 68.0
  Min: 10
  Max: 491

Dataset Statistics:
  train_samples: 4353
  train_avg_docs_per_question: 4.160578911095796
  train_avg_answer_length: 73.33195497358143
  dev_samples: 948
  dev_avg_docs_per_question: 4.836497890295359
  dev_avg_answer_length: 57.857594936708864
  valid_samples: 948
  valid_avg_docs_per_question: 4.836497890295359


In [4]:
# Examine sample
sample = df_train.iloc[0]
print("Sample Question:")
print(sample['question'])
print("\nSample Answer (long-form):")
print(sample['answer'])
print(f"\nAnswer length: {len(sample['answer'].split())} words")
print(f"Number of context documents: {len(sample['docs'])}")

Sample Question:
When does the new bunk'd come out?

Sample Answer (long-form):
The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28, 2017 and episode 42 is due to come out on May 24, 2017. 

Answer length: 31 words
Number of context documents: 4


## 3. Configure RAG System for Long-Form Generation

In [5]:
# Configure for ASQA (long-form answers)
config = RAGConfig(
    # Models
    encoder_model="sentence-transformers/all-MiniLM-L6-v2",
    generator_model="google/flan-t5-large",  # Larger model for long-form
    
    # Paths
    models_dir="../models",
    output_dir="./rag_output_asqa",
    train_data_path=train_path,
    valid_data_path=dev_path,
    
    # Retriever training
    retriever_epochs=5,
    retriever_batch_size=16,
    retriever_lr=2e-5,
    retriever_patience=3,
    
    # Generator training (adjusted for long-form)
    generator_epochs=3,
    generator_batch_size=2,  # Smaller batch for longer sequences
    generator_lr=5e-5,
    generator_max_input_tokens=512,  # Correct parameter name
    generator_max_target_tokens=512,  # Correct parameter name - longer outputs for ASQA
    
    # Retrieval
    top_k=5,
    
    # Generation
    max_new_tokens=256,  # For inference
    
    # Device
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Configuration:")
print(f"  Encoder: {config.encoder_model}")
print(f"  Generator: {config.generator_model}")
print(f"  Max target tokens: {config.generator_max_target_tokens}")
print(f"  Device: {config.device}")

Configuration:
  Encoder: sentence-transformers/all-MiniLM-L6-v2
  Generator: google/flan-t5-large
  Max target tokens: 512
  Device: cpu


## 4. Initialize RAG System

In [6]:
# Initialize RAG system
rag = RAGSystem(config=config)

# Use ASQA loader instead of default HotpotQA loader
rag.data_loader = loader


Initializing RAG System
Device: cpu
Encoder: sentence-transformers/all-MiniLM-L6-v2
Generator: google/flan-t5-large
Models directory: ..\models
Output directory: rag_output_asqa
Model cache directory: c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\models

Loading models...
Loaded from cache: sentence-transformers--all-MiniLM-L6-v2
Loaded from cache: google--flan-t5-large
Models loaded successfully!



## 5. Train Retriever Model

In [7]:
# Create retriever training examples
retriever_examples = loader.create_retriever_examples()
print(f"Created {len(retriever_examples)} retriever training examples")

Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 18677.55it/s]

Created 3450 retriever training examples
Created 3450 retriever training examples


In [8]:
# Train retriever
print("Training retriever on ASQA dataset...")
rag.train_retriever(
    epochs=config.retriever_epochs,
    batch_size=config.retriever_batch_size,
    lr=config.retriever_lr
)

Training retriever on ASQA dataset...

Training Retriever
Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 18743.31it/s]
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\training\retriever_trainer.py:146: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_fp16)
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


Created 3450 retriever training examples
Retriever trainer initialized on cpu
Created DataLoader: 3450 examples, batch_size=16

Starting retriever training...
   Epochs: 5
   Learning rate: 2e-05
   Temperature: 20.0
   Accumulation steps: 2
   Mixed precision: True


Epoch 1/5:   0%|          | 0/215 [00:00<?, ?it/s]c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\training\retriever_trainer.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_fp16):
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\torch\amp\autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch 1/5: 100%|██████████| 215/215 [05:03<00:00,  1.41s/it, loss=0.0001]


Epoch 1/5: avg_loss=0.0058
Saved best model to ..\models\retriever_trained


...

Epoch 3/5: avg_loss=0.0018
Saved best model to ..\models\retriever_trained


Epoch 4/5: 100%|██████████| 215/215 [04:56<00:00,  1.38s/it, loss=0.0003]


Epoch 4/5: avg_loss=0.0020
No improvement. Patience: 1/3


Epoch 5/5: 100%|██████████| 215/215 [04:55<00:00,  1.37s/it, loss=0.0001]

Epoch 5/5: avg_loss=0.0022
No improvement. Patience: 2/3

Training complete! Best loss: 0.0018
Model saved to: ..\models\retriever_trained



## 6. Evaluate Retriever

In [9]:
# Evaluate retriever on dev set
print("Evaluating retriever...")
retriever_metrics = rag.evaluate_retriever()

print("\nRetriever Metrics:")
for metric, value in retriever_metrics.items():
    print(f"  {metric}: {value:.4f}")

Evaluating retriever...

Evaluating Retriever

Evaluating retriever on 948 questions...
   Top-K values: [1, 3, 5]
Building corpus from validation set...
   Corpus size: 2010 unique documents
Encoding corpus (2010 documents)...


Batches: 100%|██████████| 32/32 [00:19<00:00,  1.66it/s]


Cached corpus embeddings to rag_output_asqa\corpus_embeds.pt

Evaluating retrieval performance...


Evaluating: 100%|██████████| 948/948 [00:10<00:00, 94.04it/s]


Retriever Evaluation Results:
   recall@1            : 0.6319
   precision@1         : 0.6319
   recall@3            : 0.7890
   precision@3         : 0.3678
   recall@5            : 0.8270
   precision@5         : 0.2464
   mrr                 : 0.7119


Retriever Metrics:
  recall@1: 0.6319
  precision@1: 0.6319
  recall@3: 0.7890
  precision@3: 0.3678
  recall@5: 0.8270
  precision@5: 0.2464
  mrr: 0.7119


## 7. Build FAISS Index

In [10]:
# Build corpus and index
print("Building corpus and FAISS index...")
rag.build_index()

Building corpus and FAISS index...

Building FAISS Index
Built corpus: 6269 unique passages from 4353 questions
Building FAISS index from 6269 documents...


Encoding: 100%|██████████| 98/98 [01:22<00:00,  1.18it/s]

Built FAISS index: 6269 documents, 384 dimensions
Saved index to rag_output_asqa\index
   - index.faiss: rag_output_asqa\index\index.faiss
   - meta.json: rag_output_asqa\index\meta.json



## 8. Train Generator Model

In [ ]:
# Create generator training examples
generator_examples = loader.create_generator_examples(max_examples=5000)
print(f"Created {len(generator_examples)} generator training examples")

Creating generator training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 30866.59it/s]

Created 4353 generator training examples
Created 4353 generator training examples


: 

In [ ]:
# Train generator
print("Training generator on ASQA dataset...")
print("Note: This may take several hours for long-form generation")

rag.train_generator(
    epochs=config.generator_epochs,
    batch_size=config.generator_batch_size,
    lr=config.generator_lr,
    max_examples=len(generator_examples)
)

Training generator on ASQA dataset...
Note: This may take several hours for long-form generation

Training Generator
Creating generator training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 34679.54it/s]


Created 4353 generator training examples
Generator trainer initialized on cpu
Created DataLoader: 4353 examples, batch_size=2

Starting generator training...
   Epochs: 3
   Learning rate: 5e-05
   Warmup ratio: 0.1
   Gradient accumulation: 1
   Max input tokens: 512
   Max target tokens: 512


Epoch 1/3:   0%|          | 0/2177 [00:00<?, ?it/s]

## 9. Baseline 1: Normal RAG Inference

In [ ]:
# Test on a sample question
test_question = df_dev.iloc[0]['question']
print(f"Test Question: {test_question}")

answer, contexts = rag.answer(test_question, return_contexts=True)
print(f"\nGenerated Answer:\n{answer}")
print(f"\nRetrieved {len(contexts)} contexts")

In [ ]:
# Run inference on full dev set
print("Running Normal RAG inference on dev set...")

normal_rag_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Normal RAG"):
    question = row['question']
    gold_answer = row['answer']
    
    try:
        # Generate answer
        predicted_answer, contexts = rag.answer(question, return_contexts=True)
        
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'contexts': contexts,
            'num_contexts': len(contexts)
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'contexts': [],
            'num_contexts': 0
        })

# Save results
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

df_normal_rag = pd.DataFrame(normal_rag_results)
df_normal_rag.to_csv(results_dir / "asqa_normal_rag_predictions.csv", index=False)
print(f"\n✅ Saved Normal RAG results: {len(df_normal_rag)} predictions")

## 10. Baseline 2: RAG + LLM Filter

In [ ]:
# Initialize LLM filter pipeline
filter_pipeline = LLMFilterPipeline(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-2.5-flash",
    context_threshold=6.0,
    answer_threshold=6.0,
    max_concurrent=10
)

print("Initialized LLM filter pipeline")

In [ ]:
# Test filtering on a sample
test_question = df_dev.iloc[0]['question']
test_answer, test_contexts = rag.answer(test_question, return_contexts=True)

print(f"Question: {test_question}")
print(f"\nOriginal contexts: {len(test_contexts)}")

# Filter contexts
filtered_contexts, context_results = filter_pipeline.filter_contexts(test_question, test_contexts)
print(f"Filtered contexts: {len(filtered_contexts)}")

# Show context filtering results
for i, result in enumerate(context_results):
    print(f"\nContext {i+1}: Score={result.score:.1f}, Relevant={result.is_relevant}")
    print(f"  Reasoning: {result.reasoning}")

# Filter answer
answer_result = filter_pipeline.filter_answer(test_question, test_answer, test_contexts)
print(f"\nAnswer Evaluation:")
print(f"  Faithfulness: {answer_result.faithfulness_score:.1f}")
print(f"  Relevance: {answer_result.relevance_score:.1f}")
print(f"  Completeness: {answer_result.completeness_score:.1f}")
print(f"  Overall: {answer_result.overall_quality}")
print(f"  Reasoning: {answer_result.reasoning}")

In [ ]:
# Run filtered RAG on dev set
print("Running RAG + LLM Filter on dev set...")
print("Note: This will take longer due to LLM filtering")

filtered_rag_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Filtered RAG"):
    question = row['question']
    gold_answer = row['answer']
    
    try:
        # Step 1: Retrieve contexts
        contexts, _, _ = rag.indexer.search(question, top_k=config.top_k)
        
        # Step 2: Filter contexts
        filtered_contexts, context_filter_results = filter_pipeline.filter_contexts(
            question, contexts
        )
        
        # Step 3: Generate answer with filtered contexts
        if filtered_contexts:
            qa_pipeline = rag.create_qa_pipeline()
            # Use filtered contexts for generation
            predicted_answer = qa_pipeline._generate_answer(question, filtered_contexts)
        else:
            # Fallback to all contexts if all filtered out
            predicted_answer = rag.answer(question)
            filtered_contexts = contexts
        
        # Step 4: Filter answer
        answer_filter_result = filter_pipeline.filter_answer(
            question, predicted_answer, filtered_contexts
        )
        
        filtered_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'contexts': contexts,
            'filtered_contexts': filtered_contexts,
            'num_contexts': len(contexts),
            'num_filtered_contexts': len(filtered_contexts),
            'answer_quality': answer_filter_result.overall_quality,
            'faithfulness_score': answer_filter_result.faithfulness_score,
            'relevance_score': answer_filter_result.relevance_score,
            'completeness_score': answer_filter_result.completeness_score,
            'filter_reasoning': answer_filter_result.reasoning
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        filtered_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'contexts': [],
            'filtered_contexts': [],
            'num_contexts': 0,
            'num_filtered_contexts': 0,
            'answer_quality': 'BAD',
            'faithfulness_score': 0.0,
            'relevance_score': 0.0,
            'completeness_score': 0.0,
            'filter_reasoning': f"Error: {str(e)}"
        })

# Save results
df_filtered_rag = pd.DataFrame(filtered_rag_results)
df_filtered_rag.to_csv(results_dir / "asqa_filtered_rag_predictions.csv", index=False)
print(f"\n✅ Saved Filtered RAG results: {len(df_filtered_rag)} predictions")

## 11. Quick Comparison

In [ ]:
# Compare filtering statistics
print("Filtering Statistics:")
print(f"\nContext Filtering:")
total_contexts = df_filtered_rag['num_contexts'].sum()
filtered_contexts = df_filtered_rag['num_filtered_contexts'].sum()
filter_rate = (total_contexts - filtered_contexts) / total_contexts * 100
print(f"  Total contexts retrieved: {total_contexts}")
print(f"  Contexts after filtering: {filtered_contexts}")
print(f"  Filter rate: {filter_rate:.1f}%")

print(f"\nAnswer Quality Distribution:")
quality_counts = df_filtered_rag['answer_quality'].value_counts()
print(quality_counts)
print(f"\nGood answers: {quality_counts.get('GOOD', 0) / len(df_filtered_rag) * 100:.1f}%")
print(f"Bad answers: {quality_counts.get('BAD', 0) / len(df_filtered_rag) * 100:.1f}%")

print(f"\nAverage Scores (Filtered RAG):")
print(f"  Faithfulness: {df_filtered_rag['faithfulness_score'].mean():.2f}")
print(f"  Relevance: {df_filtered_rag['relevance_score'].mean():.2f}")
print(f"  Completeness: {df_filtered_rag['completeness_score'].mean():.2f}")

## 12. Sample Outputs Comparison

In [ ]:
# Compare a few examples side by side
for i in range(min(3, len(df_normal_rag))):
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print(f"{'='*80}")
    
    print(f"\nQuestion: {df_normal_rag.iloc[i]['question']}")
    
    print(f"\nGold Answer:\n{df_normal_rag.iloc[i]['gold_answer']}")
    
    print(f"\nNormal RAG Answer:\n{df_normal_rag.iloc[i]['predicted_answer']}")
    
    print(f"\nFiltered RAG Answer:\n{df_filtered_rag.iloc[i]['predicted_answer']}")
    print(f"Quality: {df_filtered_rag.iloc[i]['answer_quality']}")
    print(f"Scores: F={df_filtered_rag.iloc[i]['faithfulness_score']:.1f}, "
          f"R={df_filtered_rag.iloc[i]['relevance_score']:.1f}, "
          f"C={df_filtered_rag.iloc[i]['completeness_score']:.1f}")

## Summary

Successfully implemented and compared two RAG baselines on ASQA:

1. **Normal RAG**: Trained retriever + generator on ASQA long-form QA
2. **RAG + LLM Filter**: Added two-stage filtering (context + answer)

Next steps:
- Generate synthetic labeled data for training
- Comprehensive evaluation with RAGAS metrics
- Detailed analysis of filter effectiveness